In [1]:
!pip install langchain langchain-community langchain-core chromadb sentence-transformers transformers accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.2/467.2 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 77.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 84.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings


model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "cuda"}

# try to access the sentence transformers from HuggingFace: https://huggingface.co/api/models/sentence-transformers/all-mpnet-base-v2
try:
    embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)
except Exception as ex:
    print("Exception: ", ex)
    # alternatively, we will access the embeddings models locally
    local_model_path = "/kaggle/input/sentence-transformers/minilm-l6-v2/all-MiniLM-L6-v2"
    print(f"Use alternative (local) model: {local_model_path}\n")
    embeddings = HuggingFaceEmbeddings(model_name=local_model_path, model_kwargs=model_kwargs)

/tmp/ipykernel_36/1551073730.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)
2025-10-23 16:09:38.529899: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761235778.775101      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761235778.855532      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin 

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
import pandas as pd
from langchain_core.documents import Document

documents = []
# <-- Đặt đường dẫn file tri thức của bạn ở đây
knowledge_base_path = '/kaggle/input/ielts-dataset/train_final.csv'

try:
    df_knowledge = pd.read_csv(knowledge_base_path)
    print(f"Đã đọc thành công file tri thức. Tổng số ví dụ: {len(df_knowledge)}")

    # Chuyển đổi mỗi hàng trong DataFrame thành một đối tượng LangChain Document
    for index, row in df_knowledge.iterrows():
        # Kết hợp các phần văn bản quan trọng để tạo ngữ cảnh
        page_content = (
            f"Prompt: {row['prompt']}\n\n"
            f"Essay: {row['essay']}\n\n"
            f"Overall_Band: {row['Overall_Band']}")
        
        # Metadata bây giờ chỉ chứa thông tin về điểm số (band)
        metadata = {
            "Overall_Band": row['Overall_Band']
        }
        documents.append(Document(page_content=page_content, metadata=metadata))

    print(f"Đã tạo {len(documents)} documents từ dữ liệu tri thức.")

except FileNotFoundError:
    print(f"LỖI: Không tìm thấy file tri thức tại '{knowledge_base_path}'. Vui lòng kiểm tra lại đường dẫn.")
    documents = [] # Đảm bảo documents là list rỗng để không bị lỗi ở các bước sau
except Exception as e:
    print(f"Đã xảy ra lỗi khi đọc file tri thức: {e}")
    documents = []

Đã đọc thành công file tri thức. Tổng số ví dụ: 9833
Đã tạo 9833 documents từ dữ liệu tri thức.


In [4]:
from langchain_community.vectorstores import Chroma
vectordb = Chroma.from_documents(documents=documents, embedding=embeddings, persist_directory="chroma_db")

In [5]:
!pip install -U bitsandbytes accelerate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 28.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 24.2 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.8.1
    Uninstalling accelerate-1.8.1:
      Successfully uninstalled accelerate-1.8.1


In [6]:
import sys, gc, importlib

# --- Dọn cache của bitsandbytes và transformers ---
for m in list(sys.modules.keys()):
    if "bitsandbytes" in m or "transformers" in m:
        del sys.modules[m]

# --- Xóa cache import ---
importlib.invalidate_caches()

# --- Dọn bộ nhớ Python ---
gc.collect()

print("✅ Cleared cached modules and memory!")


✅ Cleared cached modules and memory!


In [7]:
# --- Làm sạch cache import ---
# 1️⃣ Import bitsandbytes trước
import bitsandbytes as bnb
print("bitsandbytes version:", bnb.__version__)

# 2️⃣ Sau đó import transformers và phần còn lại
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from peft import PeftModel
from langchain_community.llms import HuggingFacePipeline
import torch


# Sau đó chạy lại phần load model của bạn


# --- Đường dẫn model ---
BASE_MODEL = "unsloth/meta-llama-3.1-8b-bnb-4bit"
LORA_ADAPTER_PATH = "/kaggle/input/llma_8b/pytorch/default/1"   # đổi sang repo id hoặc đường dẫn LoRA bạn đã fine-tune

# --- Cấu hình load 4-bit ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",     # nf4 cho chất lượng tốt
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# --- Load tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

# --- Load base model (4bit) ---
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

# --- Áp dụng LoRA adapter ---
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_PATH)
model.eval()

# --- Tạo pipeline text generation ---
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto",
    max_new_tokens=512,
    temperature=0.1,
    top_p=0.9,
)

# --- Gói vào LangChain LLM ---
llm = HuggingFacePipeline(pipeline=generator)

print("✅ LoRA (PEFT) model loaded successfully with 4-bit quantization!")

bitsandbytes version: 0.48.1


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

Device set to use cuda:0
Device set to use cuda:0


✅ LoRA (PEFT) model loaded successfully with 4-bit quantization!


/tmp/ipykernel_36/790338008.py:55: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=generator)


In [18]:
!pip install langchain==0.3.0

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached langchain_core-0.3.79-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_text_splitters-0.3.11-py3-none-any.whl.metadata (1.8 kB)
INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 24.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.1.53
    Uninstalling langchain-core-0.1

In [40]:
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain


qa_chain = None
if documents:
    prompt_template = """
    Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

    ### Instruction:
    You are an IELTS Writing Task 2 examiner. Evaluate the essay using IELTS band descriptors. 
    Internally calculate scores for the four criteria, average them, and round to the nearest 0.5.
    Return ONLY the overall band score in strict JSON format.
    
    ### Context:    
    {context}
    
    ### Input:
    {input}
    
    ### Response format:
    Return the final overall band score as a number in format: The final answer is: $\boxed{{score}}$ or Response: {{score}}
    
    ### Response:
    """

    PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "input"])

    retriever = vectordb.as_retriever(search_kwargs={'k': 4}) # Lấy 2 ví dụ liên quan nhất

    combine_docs_chain = create_stuff_documents_chain(
        llm, PROMPT
    )
    retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)
    print("RAG chain đã sẵn sàng.")
else:
    print("Không thể tạo RAG chain vì không có dữ liệu tri thức.")

RAG chain đã sẵn sàng.


In [41]:
import json
import time
import re

def grade_ielts_essay(chain, prompt, essay, row):
    """
    Chấm IELTS essay và lấy band duy nhất.
    Ưu tiên theo thứ tự:
      1️⃣ Dạng 'The final answer is: $\\boxed{6.5}$'
      2️⃣ Dạng 'Response: 6.5'
      3️⃣ Trích từ JSON có key 'overall_band_score' hoặc tương tự.
    """
    if not chain:
        return {"error": "RAG chain is not available."}

    # ✅ chỉ dùng prompt để truy xuất ngữ cảnh từ vectordb
    retrieval_query = prompt  

    # Nhưng essay + prompt vẫn là input để đánh giá
    full_question = f"Prompt: {prompt}\n\nEssay: {essay}"

    start_time = time.time()
    result = chain.invoke({"input": full_question})
    end_time = time.time()
    response_time = round(end_time - start_time, 2)

    rag_response_str = str(result.get("result", "")).strip()
    print(rag_response_str)

    band_score = None  # mặc định nếu không tìm thấy

    # 1️⃣ Ưu tiên dạng "The final answer is: $\boxed{6.5}$"
    pattern_boxed = r"The\s*final\s*answer\s*is\s*:\s*\$?\\?boxed\{?\$?([0-9]+(?:\.[0-9]+)?)\$?\}?"
    match_boxed = re.search(pattern_boxed, rag_response_str, flags=re.IGNORECASE)
    if match_boxed:
        band_score = float(match_boxed.group(1))

    # 2️⃣ Nếu không có, thử dạng "Response: 6.5"
    if band_score is None:
        pattern_response = r"Response\s*:\s*([0-9]+(?:\.[0-9]+)?)"
        match_response = re.search(pattern_response, rag_response_str, flags=re.IGNORECASE)
        if match_response:
            band_score = float(match_response.group(1))

    # 3️⃣ Nếu vẫn chưa có, thử tìm trong JSON
    if band_score is None:
        try:
            json_pattern = r"\{[\s\S]*?\}"
            json_match = re.search(json_pattern, rag_response_str)
            if json_match:
                json_str = json_match.group(0)
                data = json.loads(json_str)
                # tìm các key có thể chứa điểm
                for key in data.keys():
                    if "band" in key.lower() or "score" in key.lower():
                        band_score = float(data[key])
                        break
        except Exception as e:
            return {
                "error": f"Không trích xuất được band score: {e}",
                "raw_response": rag_response_str
            }

    # 4️⃣ Nếu vẫn không tìm thấy
    if band_score is None:
        return {
            "error": "Không tìm thấy band score trong response.",
            "raw_response": rag_response_str
        }

    return {
        "predicted_band": band_score,
        "actual_band": row.get("Overall_Band", None),
        "response_time": response_time
    }


In [ ]:
from IPython.display import display, Markdown

# %% [markdown]
# ---
# ## Phần 5: Chạy Inference trên toàn bộ file test

# %% [code]
test_file_path = '/kaggle/input/ielts-dataset/test_final.csv'
all_results = []

try:
    df_test = pd.read_csv(test_file_path).head(2)
    print(f"Đã đọc thành công file test. Tổng số bài luận cần chấm: {len(df_test)}")
    
    for index, row in df_test.iterrows():
        prompt = row['prompt']
        essay = row['essay']
        
        display(Markdown(f"### 📝 Đang chấm bài luận #{index + 1}..."))
        display(Markdown(f"**Prompt:** {prompt[:100]}..."))
        
        grading_result = grade_ielts_essay(retrieval_chain, prompt, essay, row)
        
        display(Markdown("#### ✅ Kết quả chấm điểm:"))
        display(Markdown(f"```json\n{json.dumps(grading_result, indent=2)}\n```"))
        
        all_results.append(grading_result)
        
        display(Markdown("---"))

except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file test tại đường dẫn '{test_file_path}'")
except Exception as e:
    print(f"Đã xảy ra lỗi không mong muốn: {e}")

Đã đọc thành công file test. Tổng số bài luận cần chấm: 2


### 📝 Đang chấm bài luận #1...

**Prompt:** Some people think that the best way to solve global environmental problems is to increase the cost o...

In [ ]:
# %% [markdown]
# ---
# ## Phần 6: Lưu kết quả

# %% [code]
if all_results:
    output_json_path = 'grading_results.json'
    with open(output_json_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f"Đã lưu thành công {len(all_results)} kết quả vào file '{output_json_path}'")

    output_csv_path = 'grading_results.csv'
    df_results = pd.DataFrame(all_results)
    df_results.to_csv(output_csv_path, index=False)
    print(f"Đã lưu thành công {len(all_results)} kết quả vào file '{output_csv_path}'")

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, mean_absolute_error
import numpy as np

# --- Đọc file ---
df = pd.read_csv("grading_results.csv")  # đổi tên file cho phù hợp

# --- Làm sạch nhẹ ---
def clean_text(x):
    if isinstance(x, str):
        return x.strip().replace('\r', '').replace('\n', '')
    return str(x).strip()

df["predicted_band"] = df["predicted_band"].apply(clean_text)
df["actual_band"] = df["actual_band"].apply(clean_text)

# --- Accuracy và F1 ---
accuracy = accuracy_score(df["actual_band"], df["predicted_band"])
f1 = f1_score(df["actual_band"], df["predicted_band"], average="macro")

# --- RMSE & MAE ---
def safe_float(x):
    try:
        return float(x.replace("<", ""))  # "<4" → 4.0
    except:
        return np.nan

df["pred_num"] = df["predicted_band"].apply(safe_float)
df["true_num"] = df["actual_band"].apply(safe_float)

valid = df.dropna(subset=["pred_num", "true_num"])

rmse = mean_squared_error(valid["true_num"], valid["pred_num"], squared=False)
mae = mean_absolute_error(valid["true_num"], valid["pred_num"])

# --- In kết quả ---
print("===== Evaluation Metrics =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"RMSE     : {rmse:.4f}")
print(f"MAE      : {mae:.4f}")

# --- Lưu vào file CSV ---
results = pd.DataFrame([{
    "Accuracy": round(accuracy, 4),
    "F1_Score": round(f1, 4),
    "RMSE": round(rmse, 4),
    "MAE": round(mae, 4),
    "Total_Samples": len(df),
    "Valid_Samples_for_RMSE": len(valid)
}])

results.to_csv("evaluation_results.csv", index=False)
print("\n✅ Saved results to 'evaluation_results.csv'")